In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

In [2]:
BASE = "https://www.christerhamp.se/runor/gamla/"
INDEX_URL = urljoin(BASE, "index.html")

# Один session на всё, с нормальным User-Agent
SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0 Safari/537.36"
    )
})

In [3]:

def get_soup(url: str, max_retries: int = 5) -> BeautifulSoup:
    for attempt in range(max_retries):
        resp = SESSION.get(url)
        status = resp.status_code

        if status == 429:
            wait = 5 * (attempt + 1) + random.uniform(0, 3)
            print(f"429 на {url}, попытка {attempt + 1}/{max_retries}, пауза {wait:.1f} сек")
            time.sleep(wait)
            continue

        resp.raise_for_status()

        # Ключевой блок: аккуратно выбираем кодировку
        # 1) если сервер отдал charset — используем его
        # 2) если отдал дефолт latin-1 — заменяем на apparent_encoding
        # 3) если вообще ничего — тоже apparent_encoding
        enc = resp.encoding
        if not enc or enc.lower() in ("iso-8859-1", "latin-1"):
            if resp.apparent_encoding:
                enc = resp.apparent_encoding
        resp.encoding = enc or "utf-8"

        return BeautifulSoup(resp.text, "html.parser")

    raise requests.HTTPError(f"Too many 429 for {url}")


def collect_inscription_urls(index_url: str = INDEX_URL) -> list[str]:
    soup = get_soup(index_url)
    urls = []

    for a in soup.find_all("a", href=True):
        href = a["href"]
        text = (a.get_text() or "").strip()

        # только локальные html
        if href.startswith("http"):
            continue
        if not href.lower().endswith(".html"):
            continue
        if "index" in href.lower():
            continue

        # отбрасываем навигацию
        if any(x in text for x in ["Tillbaka", "Åter till", "Rundata"]):
            continue

        full_url = urljoin(BASE, href)
        urls.append(full_url)

    # уникализация с сохранением порядка
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def extract_block(soup: BeautifulSoup, cls: str) -> str | None:
    """
    Берём ТОЛЬКО прямой текст внутри <p class="cls">,
    игнорируя вложенные теги (типа <p class="overs"> внутри <p class="trans">).
    """
    ps = soup.find_all("p", class_=cls)
    chunks = []
    for p in ps:
        # только прямые текстовые дети, без рекурсии
        direct_texts = [t.strip() for t in p.find_all(string=True, recursive=False)]
        txt = " ".join(t for t in direct_texts if t)
        if txt:
            chunks.append(txt)
    if not chunks:
        return None
    return "\n".join(chunks)


def find_full_image_url(soup: BeautifulSoup, page_url: str) -> str | None:
    # Сначала ищем ссылку "Större bild"
    a = soup.find("a", string=lambda s: s and "Större bild" in s)
    if a and a.has_attr("href"):
        return urljoin(page_url, a["href"])

    # fallback — первая картинка
    img = soup.find("img")
    if img and img.get("src"):
        return urljoin(page_url, img["src"])

    return None


def parse_page(url: str) -> dict:
    soup = get_soup(url)

    # title → имя памятника
    title_tag = soup.find("title")
    title_text = title_tag.get_text(strip=True) if title_tag else url

    # stone_id = всё до " - " из title
    stone_id = title_text.split(" - ", 1)[0].strip()

    runic_text = extract_block(soup, "run")       # <p class="run">
    translit = extract_block(soup, "trans")       # <p class="trans">
    overs = extract_block(soup, "overs")          # <p class="overs">

    img_url = find_full_image_url(soup, url)

    return {
        "stone_id": stone_id,
        "name": title_text,
        "runic": runic_text,                      # рунический текст (если задан)
        "transliteration": translit,              # латинская транслитерация
        "translation": overs,                     # перевод/нормализованный текст
        "page_url": url,
        "image_url": img_url,
    }


def build_dataframe() -> pd.DataFrame:
    urls = collect_inscription_urls()
    rows = []

    for i, url in enumerate(urls, 1):
        print(f"[{i}/{len(urls)}] {url}")
        try:
            row = parse_page(url)
            rows.append(row)
        except Exception as e:
            print(f"Ошибка на {url}: {e}")

    df = pd.DataFrame(rows)
    return df

In [4]:
df = build_dataframe()
print(df[["stone_id", "runic", "transliteration", "translation"]].head())
df.to_csv("christerhamp_gamla_runor.csv", index=False)

[1/2945] https://www.christerhamp.se/runor/hnias.html
[2/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl1.html
[3/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl3.html
[4/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl4.html
[5/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl5.html
[6/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl6.html
[7/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl8.html
[8/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl10.html
[9/2945] https://www.christerhamp.se/runor/gamla/dk/dkbl12.html
[10/2945] https://www.christerhamp.se/runor/gamla/dr/drbr75.html
[11/2945] https://www.christerhamp.se/runor/gamla/bo/bohoga.html
[12/2945] https://www.christerhamp.se/runor/gamla/bo/bokalleby.html
[13/2945] https://www.christerhamp.se/runor/gamla/bo/boniyr1.html
[14/2945] https://www.christerhamp.se/runor/gamla/bo/boniyr2.html
[15/2945] https://www.christerhamp.se/runor/gamla/bo/boniyr3.html
[16/2945] https://www.christerhamp.se/runor/